# Generating random two-qubit states

This notebook generates the random density matrices used to train the entanglement classifiers. It uses the functions in `matrices/modules/generate_quantum_states.py`.

**How to run it.** Run the cells from top to bottom (in Jupyter: *Kernel → Restart & Run All*). The whole notebook takes under a minute with the class settings below.

**Where the output goes.** All files are written to a new folder `demo_output/`, so the original data in `matrices/input_data/` is never overwritten.

**File format.** One density matrix per line: 32 numbers, Re ρ₁₁, Im ρ₁₁, Re ρ₁₂, Im ρ₁₂, … in row-major order. Lines starting with `#` are headers with summary statistics.

# 0. Setup

In [ ]:
import os, sys, time
import numpy as np
import matplotlib.pyplot as plt
from numpy.random import Generator, PCG64

# Find the student's modules, whether the notebook runs from the repo root or from matrices/
for candidate in ["matrices/modules", "modules", "../matrices/modules"]:
    if os.path.isfile(os.path.join(candidate, "generate_quantum_states.py")):
        MODULES = os.path.abspath(candidate)
        break
else:
    raise FileNotFoundError("Run this notebook from the repository root (the folder that contains 'matrices').")
sys.path.insert(0, MODULES)

from generate_quantum_states import (generate_random_pure_density_matrix,
                                     generate_random_mixed_density_matrix,
                                     generate_M_random_DMs_file,
                                     generate_random_DMs_files_negativitywise,
                                     random_int)
from physical_tests import negativity, purity

# All output goes here, never into matrices/input_data
OUT = os.path.abspath("demo_output")
for sub in ["pure_states", "mixed_states"]:
    os.makedirs(os.path.join(OUT, sub), exist_ok=True)
print("Modules loaded from:", MODULES)
print("Output folder:      ", OUT)

In [ ]:
# Class settings: small sizes so everything runs in seconds.
# The original data sets used M = 10000 per file (about 10 minutes for the mixed states).
SEED = 2026          # fixed seed: everyone gets the same states. Use int(time.time()) for fresh ones.
M_FILE = 1000        # states in each 'all negativities' file
M_PER_BIN = 200      # states in each negativity bin
N = 4                # Hilbert space dimension (two qubits). The negativity code only works for N = 4.

The next cell shows how the imported random number generator works. It is a generator from the numpy.random library with a period of $2^{128}$ (numpy.org/doc/stable/reference/random/bit_generators/pcg64.html#numpy.random.PCG64). This is more than enough for our needs. For example, suppose we generate 10^4 random states. With the algorithm given below, each state needs 8 random numbers, so we must generate $8\times 10^4$ random numbers, many orders of magnitude fewer than $2^{128}$.

In [ ]:
rg = Generator(PCG64(SEED))
print(rg.random())
print(rg.random(4,))

# 1. Pure states

## 1.1. Theory and algorithm

For a space $\mathcal{H}$ of dimension $N$, we can take a basis $\{|u_k\rangle ,\;\; k=1,...,N\equiv\text{dim}(\mathcal{H})\}$, so that $\forall|\psi\rangle\in\mathcal{H}$:

>\begin{equation}
|\psi\rangle = \sum _{i=1}^{N} a_i|u_i\rangle \;\;\;\text{with}\;\;\;\sum |a_i|^2 = 1 \;\;\; a_i\in\mathbb{C}
\end{equation}

In general $a_j = r_j e^{i\phi _j}$ with $r_j>0$ and $\phi_j\in[0,2\pi)$. To generate random states numerically, we take $0\leq r_j\leq 1$. Note that an upper bound on the moduli $r_j$ does not lose generality, because the state has to be normalized anyway. That is, with $0\leq r_j\leq 1$ we can sample the whole space (except the separable states; we discuss why below). For example, we can sample a state in which the amplitude of one basis state $|u_k\rangle$ is much larger than all the others. This simply requires $0\leq r_j<<r_k\leq 1$ for all $|u_j\rangle$ with $j\neq k$, so that after normalizing by $\sqrt{\langle \psi|\psi\rangle}=\sqrt{\sum r_i ^2}$ we get the desired ratio of amplitudes. In the same way, we can sample states in which all basis states have very similar amplitudes, which happens when $r_j\simeq r_i$ $\forall i,j=1,...,N$.


On the other hand, a separable state is obtained when $r_j=0$ $\forall j\neq k$. Since such states require an exact 'zero' in the moduli of the coefficients of all basis states of $\mathcal{H}$ except one, we can picture such a state as a point (a zero-dimensional manifold) embedded in the space $\mathcal{H}$. Because we generate the moduli $r_j$ and phases $\phi _j$ with a (numerical) random number generator, we will never sample an exact zero, so in practice we will never obtain a separable state. This is not a problem for us. When I need separable states, I will take them from the state files I already have, or generate new ones with an algorithm that takes this detail into account, i.e. one that sets the modulus to zero for all basis states except one.

Going back to the reasoning of two paragraphs ago, you can check that a state $|\psi\rangle$ built (and normalized) this way gives the density matrix:

>\begin{equation}
|\psi\rangle\langle\psi| \doteq \frac{1}{\sum _i r_i ^2} \sum _{j,k} \tilde{c_{jk}} |u_j\rangle\langle u_k| \;\;\;\text{with}\;\;\; \tilde{c_{jk}} = r_jr_ke^{i(\phi _j -\phi _k})
\end{equation}

Therefore, the matrix representation of $|\psi\rangle\langle\psi|$ is:

>\begin{equation}
|\psi\rangle\langle\psi| \doteq \big(c_{jk}\big)_{j,k=1,...,N} \;\;\; \text{with}\;\;\; c_{jk} = \frac{\tilde{c_{jk}}}{\sum _i r_i ^2} = \frac{r_jr_ke^{i(\phi_j -\phi_k)}}{\sum _i r_i ^2}
\end{equation}

That is, $c_{jk}$ is the matrix element in row $j$ and column $k$. We have implicitly taken the basis

>\begin{equation}
|u_k\rangle = \big(\delta _{ik}\big)_{i=1,...,N}
\end{equation}

That is, $|u_1\rangle^{T}=(1,0,...,0)$.

So, to generate a random quantum state of $\mathcal{H}$ with $\text{dim}(\mathcal{H})\equiv N$, it is enough to:

1) Generate $N$ random numbers $0\leq r_i\leq 1$, $i=1,...,N$.

2) Generate $N$ random numbers $0\leq \phi_i< 2\pi$, $i=1,...,N$.

3) Build the matrix representation $\big(c_{jk}\big)_{j,k=1,...,N}$ from the definition above.

## 1.2. Look at one state

Before generating thousands of states, check one by hand. A valid density matrix is Hermitian, has trace 1 and has no negative eigenvalues. For a pure state the purity Tr ρ² equals 1. The negativity is 0 for separable states and at most 0.5 for two qubits.

In [ ]:
rng = Generator(PCG64(SEED))
rho = generate_random_pure_density_matrix(N, rng)
np.set_printoptions(precision=3, suppress=True)
print("rho =\n", rho)
print("Hermitian:  ", np.allclose(rho, rho.conj().T))
print("Trace:      ", round(np.trace(rho).real, 6))
print("Eigenvalues:", np.linalg.eigvalsh(rho).round(6))
print("Purity:     ", round(purity(N, rho), 6))
print("Negativity: ", round(negativity(rho, N), 6))

In [ ]:
# Sanity check on two states whose negativity we know
product = np.zeros((4, 4), dtype=complex); product[0, 0] = 1               # |00><00|
bell = np.array([[1,0,0,1],[0,0,0,0],[0,0,0,0],[1,0,0,1]], dtype=complex)/2   # |Phi+><Phi+|
print("Negativity of |00>:  ", negativity(product, N))
print("Negativity of |Phi+>:", negativity(bell, N))

In [ ]:
# How are the random pure states distributed in negativity?
rng = Generator(PCG64(SEED))
negs = np.array([negativity(generate_random_pure_density_matrix(N, rng), N) for _ in range(5000)])
plt.hist(negs, bins=50, range=(0, 0.5))
plt.xlabel("Negativity"); plt.ylabel("Number of states")
plt.title("5000 random pure states")
plt.show()
print(f"Mean negativity {negs.mean():.3f}; fraction with N < 0.1: {(negs < 0.1).mean():.3f}")

## 1.3. Generate a single file with random pure states, regardless of their negativity

In [ ]:
%%time
rng = Generator(PCG64(SEED))
filepath = os.path.join(OUT, "pure_states", "all_negativities.txt")
negativity_mean, negativity_std = generate_M_random_DMs_file(N, M_FILE, filepath, rng, pure=True)
print(f"Wrote {M_FILE} states to {filepath}")
print(f"Negativity: mean {negativity_mean:.4f}, std {negativity_std:.4f}")

## 1.4. Generate one file per negativity bin

The function keeps drawing random states and sorts each into a bin of width `neg_resolution`, until every bin holds `M_PER_BIN` states. Separable states (negativity ≈ 0) are discarded.

In [ ]:
%%time
rng = Generator(PCG64(SEED))
filepath_root = os.path.join(OUT, "pure_states", "negativity")
negativities_mean, negativities_std = generate_random_DMs_files_negativitywise(
    0.1, M_PER_BIN, N, filepath_root, rng, max_neg=0.5, pure=True)
print("Mean negativity per bin:", np.round(negativities_mean, 3))
print("Std per bin:            ", np.round(negativities_std, 3))

# 2. Mixed states

## 2.1. Theory and algorithm

Once we are able to generate random pure density matrices it is straightforward to generate mixed ones, just by taking convex combinations of the generated random pure density matrices. We shall first fix some integer L>1, so that a mixed states is a linear convex combination

>\begin{equation}
\rho = \sum _{i=1}^L c_i|\psi _i\rangle\langle\psi _i| \;\;\;\;\;\; \sum_{i=1}^L c_i = 1
\end{equation}

In order to generate such random mixed states I will fix some integer $L_{max}$, so that I iteratively generate random integers $L$ with $2<L<L_{max}$. For a given integer, $L$, I generate $L$ random numbers $0\leq \tilde{c_i}\leq1$ and $L$ random pure states, $|\psi _i\rangle\langle\psi _i|$. Then, I build the previous convex linear combination, where:

>\begin{equation}
c_i = \frac{\tilde{c_i}}{\sum _{j=1}^L \tilde{c_j}} \;\;\;\forall i
\end{equation}

## 2.2. Purity and entanglement against L

The more pure states we mix, the closer ρ gets to the maximally mixed state 𝟙/4. Its purity drops towards 1/4, and fewer states stay entangled.

In [ ]:
rng = Generator(PCG64(SEED))
Ls = range(2, 10)
mean_purity, frac_entangled = [], []
for L in Ls:
    states = [generate_random_mixed_density_matrix(N, L, rng) for _ in range(500)]
    mean_purity.append(np.mean([purity(N, r) for r in states]))
    frac_entangled.append(np.mean([negativity(r, N) > 1e-10 for r in states]))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(Ls, mean_purity, "o-"); ax[0].axhline(0.25, ls="--", c="gray")
ax[0].set_xlabel("L (pure states mixed)"); ax[0].set_ylabel("Mean purity Tr ρ²")
ax[1].plot(Ls, frac_entangled, "o-")
ax[1].set_xlabel("L (pure states mixed)"); ax[1].set_ylabel("Fraction entangled")
plt.tight_layout(); plt.show()

## 2.3. Generate a single file with random mixed states, regardless of their negativity

In [ ]:
%%time
rng = Generator(PCG64(SEED))
filepath = os.path.join(OUT, "mixed_states", "all_negativities.txt")
negativity_mean, negativity_std, L_mean, L_std, purity_mean, purity_std = generate_M_random_DMs_file(
    N, M_FILE, filepath, rng, pure=False, L_min=2, L_max=10)
print(f"Negativity: mean {negativity_mean:.4f}, std {negativity_std:.4f}")
print(f"L:          mean {L_mean:.2f}, std {L_std:.2f}")
print(f"Purity:     mean {purity_mean:.4f}, std {purity_std:.4f}")

## 2.4. Generate one file per negativity bin

Mixed states with high negativity are rare: with L between 2 and 9, only about 0.3% of the states land in the bin (0.4, 0.5). Filling each bin with `M_PER_BIN` states takes about 10–20 seconds. With M = 10000 it takes about 10 minutes, and with L between 3 and 6 about an hour.

In [ ]:
%%time
rng = Generator(PCG64(SEED))
filepath_root = os.path.join(OUT, "mixed_states", "negativity")
negativities_mean, negativities_std, Ls_mean, Ls_std, purities_mean, purities_std = generate_random_DMs_files_negativitywise(
    0.1, M_PER_BIN, N, filepath_root, rng, max_neg=0.5, pure=False, L_min=2, L_max=10)
print("Mean negativity per bin:", np.round(negativities_mean, 3))
print("Mean L per bin:         ", np.round(Ls_mean, 2))
print("Mean purity per bin:    ", np.round(purities_mean, 3))

# 3. Check the files

Read every generated file back and confirm that each state is a valid density matrix and has the negativity its file name promises.

In [ ]:
def load_states(path):
    rows = [np.array(line.split(), float) for line in open(path)
            if line.strip() and not line.startswith("#")]
    rows = np.array(rows)
    return (rows[:, ::2] + 1j*rows[:, 1::2]).reshape(-1, 4, 4)

for kind in ["pure_states", "mixed_states"]:
    for name in sorted(os.listdir(os.path.join(OUT, kind))):
        states = load_states(os.path.join(OUT, kind, name))
        negs = np.array([negativity(r, N) for r in states])
        pur = np.array([purity(N, r) for r in states])
        valid = all(np.allclose(r, r.conj().T, atol=1e-6) and abs(np.trace(r) - 1) < 1e-5
                    and np.linalg.eigvalsh(r).min() > -1e-6 for r in states)
        print(f"{kind}/{name:28s} {len(states):5d} states  valid={valid}  "
              f"negativity [{negs.min():.3f}, {negs.max():.3f}]  purity [{pur.min():.3f}, {pur.max():.3f}]")